# Day 3 — Contract Clause Search

**Pattern:** Retrieval-augmented generation (RAG) with FAISS  
**Domain:** Legal  
**Course reference:** Lab 3 (part 1)

---

In [ ]:
# Core packages (current versions). sentence-transformers pulls in PyTorch.
%pip install -q -U sentence-transformers faiss-cpu "numpy>=2"
print("Setup complete. If an import fails right after install, restart the kernel and re-run.")

## What you're building today

Legal teams spend a huge fraction of their time searching contracts for specific clauses: 'find every NDA we signed with a non-compete longer than 12 months'. Keyword search misses synonyms ('restraint of trade', 'non-solicitation'). Today we build a semantic search engine over a contract library using embeddings and FAISS, the foundation of every RAG system.

## Learning objectives

By the end of today, you will have:

- Understand what an embedding is and why semantic search beats keyword search.
- Chunk documents intelligently — overlap, sentence boundaries, metadata.
- Embed chunks and store them in a FAISS index.
- Query the index and return the most semantically similar chunks.
- See where pure retrieval breaks and why we'll need generation on top (Day 4).


## Prerequisites

- 5-10 sample contract documents (PDFs or plain text). If you don't have any, use public templates from a site like SEC EDGAR.
- pip packages: faiss-cpu, sentence-transformers (or openai for embeddings), pypdf


## Key concepts

- Embeddings: turning text into a vector of numbers that represents meaning.
- Cosine similarity: how 'close' two meanings are.
- Chunking strategies: fixed size, sentence-based, semantic.
- Why metadata (document name, section, page) is as important as the chunk text.

## Step 1 — Load and chunk the contracts

Read each contract, split it into chunks of ~500 tokens with 50-token overlap. Attach metadata to each chunk: source document, approximate page, position in doc.

In [ ]:
import os, glob

CONTRACTS_DIR = "contracts"


def strip_header(raw):
    # Files start with metadata lines, then a row of "====", then the contract body.
    if "=" * 10 in raw:
        return raw.split("=" * 10, 1)[1].lstrip("=").lstrip()
    return raw


def load_documents(folder_path):
    documents = []
    for path in sorted(glob.glob(os.path.join(folder_path, "*.txt"))):
        with open(path, "r", encoding="utf-8") as f:
            text = strip_header(f.read())
        documents.append({"text": text, "source": os.path.basename(path), "page": 1})
    return documents


def chunk_text(text, chunk_size=500, overlap=50):
    # overlap reuses the previous chunk's tail so a clause isn't split clean in half.
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += chunk_size - overlap
    return chunks


documents = load_documents(CONTRACTS_DIR)
all_chunks = []
for doc in documents:
    for i, piece in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "text": piece,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": i,
        })

avg_len = sum(len(c["text"].split()) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(all_chunks)}")
print(f"Avg words : {avg_len:.0f} per chunk")

## Step 2 — Embed the chunks

Each chunk becomes a vector. You can use a local sentence-transformer model (all-MiniLM-L6-v2 is the standard starter — 384 dimensions, free, fast) or OpenAI's text-embedding-3-small (1536 dim, paid, higher quality).

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Same model must embed both the corpus and the queries.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")  # FAISS needs float32
print("Embeddings shape:", embeddings.shape)


# Sanity check: a confidentiality question should land nearest a confidentiality chunk.
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

probe = embedder.encode("obligation to keep information confidential", convert_to_numpy=True)
sims = sorted((cosine(probe, e), all_chunks[i]["source"]) for i, e in enumerate(embeddings))
print("Closest chunk :", sims[-1])
print("Farthest chunk:", sims[0])

## Step 3 — Build a FAISS index

FAISS is Facebook's vector similarity library — fast and free. For a starter dataset use IndexFlatIP (inner product, exact search). Later you can graduate to IVF or HNSW.

In [ ]:
import faiss

dim = embeddings.shape[1]          # 384
index = faiss.IndexFlatIP(dim)     # IP = inner product

# Normalize to length 1 so inner product == cosine similarity.
index_vectors = embeddings.copy()
faiss.normalize_L2(index_vectors)
index.add(index_vectors)

print("Vectors in index:", index.ntotal, "(should equal chunk count:", len(all_chunks), ")")

## Step 4 — Search

Take a natural-language query, embed it the same way, and ask FAISS for the top-k nearest chunks. Return the chunks plus their metadata plus their similarity scores.

In [ ]:
def search(query, k=5):
    # Embed + normalize the query the same way as the corpus, then take the k nearest.
    q = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


# Queries deliberately worded differently from the contracts.
queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
    "limitation of liability and indemnification",
    "obligation to keep information confidential",
]

for q in queries:
    print(f"\n=== {q} ===")
    for chunk, score in search(q, k=3):
        snippet = " ".join(chunk["text"].split())[:180]
        print(f"[{chunk['source']}] (score={score:.3f}) {snippet}...")

## Step 5 — Where retrieval-only breaks

You'll notice that returning chunks doesn't actually answer the lawyer's question — they still have to read them. The retrieval is necessary but not sufficient. Tomorrow we feed those chunks to an LLM and ask it to synthesize an answer with citations. That is full RAG.

In [ ]:
# Retrieval finds the right paragraph, but you still have to read it -- the gap Day 4 closes.
q = "who owns the IP if we co-develop something"
chunk, score = search(q, k=1)[0]
print(f"Question: {q}")
print(f"Top match from {chunk['source']} (score={score:.3f}), {len(chunk['text'].split())} words:\n")
print(chunk["text"])

## Things worth understanding

- What happens if your chunk size is too small (50 tokens)? Too large (2000)?
- Why does normalizing the embeddings matter when using inner-product search?
- If a single answer requires combining info from three different clauses across two contracts, can a single retrieve-top-k call find it?

## Pitfalls to avoid

- Embedding the query with a different model than the corpus — your scores become meaningless.
- No metadata — you retrieve a chunk and have no idea which document it came from. Useless in production.
- Treating cosine similarity as 'truth probability'. A high score means 'similar', not 'correct answer'.


---

## Stretch goal — rebuild this with a framework (LangChain & LlamaIndex)

You built the retrieval engine from scratch above so you understand every layer. Now re-implement the *same* thing with the two frameworks interviewers ask about. Point both at the `contracts/` folder and run the identical queries, then compare what comes back to your hand-rolled version.

> Both cells use the same local `all-MiniLM-L6-v2` embedder — no API key required.

### Stretch A — LangChain

The exact same retrieve pipeline you built by hand in Steps 1–4, collapsed into ~10 lines. LangChain hides the chunking, normalization, and FAISS index behind its abstractions — which is convenient, but notice what defaults it picks *for* you (flagged in the comments). Runs against the same `contracts/` corpus and the same queries so you can compare results directly.

In [ ]:
# Stretch A -- the same pipeline in LangChain
# pip install langchain langchain-community sentence-transformers faiss-cpu
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

docs = DirectoryLoader(
    "contracts", glob="*.txt",
    loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
).load()

# WATCH OUT: chunk_size here is in CHARACTERS, not tokens -- 500 != your Step-1 window.
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

# Same embedder as the from-scratch build, so the comparison is apples-to-apples.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)  # normalization + index handled internally

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for doc, score in db.similarity_search_with_score(q, k=3):
        src = doc.metadata.get("source", "?")
        print(f"[{src}] (dist={score:.3f}) {doc.page_content[:160].strip()}...")


### Stretch B — LlamaIndex

LlamaIndex is purpose-built for RAG/retrieval, so the same pipeline is just as short but its defaults sit closer to what you hand-rolled in Steps 1–4 (token-based, sentence-aware chunking). Run it against the same `contracts/` corpus and the same queries.

In [ ]:
# Stretch B -- the same pipeline in LlamaIndex
# pip install llama-index-core llama-index-embeddings-huggingface llama-index-readers-file
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm = None  # retrieval only -- prevents accidental OpenAI calls
# NOTE: SentenceSplitter chunk_size is in TOKENS (unlike LangChain's characters) and splits
# on sentence boundaries -- closest of the three to Step 1's intent.
Settings.node_parser = SentenceSplitter(chunk_size=500, chunk_overlap=50)

documents = SimpleDirectoryReader("contracts", required_exts=[".txt"]).load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=3)

queries = [
    "non-compete period longer than one year",
    "restrictions on hiring our employees after the contract ends",
    "who owns the IP if we co-develop something",
]
for q in queries:
    print(f"\n=== {q} ===")
    for node in retriever.retrieve(q):
        src = node.metadata.get("file_name", "?")
        print(f"[{src}] (score={node.score:.3f}) {node.text[:160].strip()}...")


## What the framework comparison teaches you

You just built the same retrieval pipeline three ways. Sit with the differences — this is the exact reflection an interviewer is probing for:

- **From scratch (Steps 1–4):** you controlled chunking, normalization, the FAISS index type, and the distance metric. ~40 lines, but you can debug every layer.
- **LangChain:** ~10 lines. Note that `RecursiveCharacterTextSplitter`'s `chunk_size` is in **characters**, not tokens — so `500` here is *not* the same window as your from-scratch token-based chunker. This is the kind of hidden default that silently changes your retrieval quality.
- **LlamaIndex:** also ~10 lines. Its `SentenceSplitter` measures `chunk_size` in **tokens** and splits on sentence boundaries by default — closer to what Step 1 asked you to do by hand. We set `Settings.llm = None` so it does pure retrieval (no accidental OpenAI calls); tomorrow's Day 4 is where you'd plug an LLM back in for full RAG.

**The interview takeaway:** lead with the from-scratch mechanics ("I chunk with overlap, normalize, use cosine via inner product..."), *then* say "in production I'd reach for LlamaIndex or LangChain to avoid reinventing it." Knowing the primitives is what lets you reason about *why* the two frameworks return different chunks for the same query — go look: do they?
